# Persian G2P VITS training

This notebook trains a Persian VITS model whose input is explicitly produced by Homo-GE2PE. The pipeline is:

`Persian text -> Homo-GE2PE -> phoneme sequence -> Coqui VITS -> waveform`

VITS never sees the original Persian graphemes. The same G2P stage is therefore required at inference time.

## 1. Install the project

Use Python 3.10–3.13. A CUDA GPU is strongly recommended for training.

In [1]:
from pathlib import Path
import os
import subprocess
import sys

ROOT = Path.cwd().resolve()
if not (ROOT / 'pyproject.toml').exists():
    ROOT = ROOT.parent
assert (ROOT / 'pyproject.toml').exists(), 'Open this notebook from the repository.'
os.chdir(ROOT)
SRC_DIR = ROOT / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))
os.environ['PYTHONPATH'] = str(SRC_DIR) + os.pathsep + os.environ.get('PYTHONPATH', '')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', '.[notebook]'], check=True)
print('Project root:', ROOT)

Looking in indexes: https://pypi.org/simple/
Obtaining file:///Users/melika/Documents/ChatGPT/persian%20TTS/persian-g2p-vits
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Checking if build backend supports build_editable: started
  Checking if build backend supports build_editable: finished with status 'done'
  Getting requirements to build editable: started
  Getting requirements to build editable: finished with status 'done'
  Preparing editable metadata (pyproject.toml): started
  Preparing editable metadata (pyproject.toml): finished with status 'done'
  Building editable for persian-g2p-vits (pyproject.toml): started
  Building editable for persian-g2p-vits (pyproject.toml): finished with status 'done'
  Created wheel for persian-g2p-vits: filename=persian_g2p_vits-0.1.0-0.editable-py3-none-any.whl size=5406 sha256=0267acd3727c27563d9e5e488392b6708a1405c07d5317a9a7d1a429dfcc276f
  Stored in directory: /private/var/folders/z


[notice] A new release of pip available: 22.3 -> 26.2.2
[notice] To update, run: pip3 install --upgrade pip


In [2]:
import torch
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
print('MPS available:', torch.backends.mps.is_available())

PyTorch: 2.2.2
CUDA available: False
MPS available: True


## 2. Verify the Persian G2P stage

The first run downloads the Homo-GE2PE checkpoint.

In [3]:
# Keep this cell usable even if the active notebook kernel was selected
# before the editable package installation completed.
from pathlib import Path
import sys

if 'ROOT' not in globals():
    ROOT = Path.cwd().resolve()
    if not (ROOT / 'pyproject.toml').exists():
        ROOT = ROOT.parent
SRC_DIR = ROOT / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from persian_g2p_vits import PersianG2P

g2p = PersianG2P()
example_text = 'سلام، این یک آزمایش تبدیل متن فارسی به گفتار است.'
example_phonemes = g2p.phonemize(example_text)
print('Text:', example_text)
print('Phonemes:', example_phonemes)

/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[transformers] Disabling PyTorch because PyTorch >= 2.5 is required but found 2.2.2
[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


ImportError: 
T5ForConditionalGeneration requires the PyTorch library but it was not found in your environment. Check out the instructions on the
installation page: https://pytorch.org/get-started/locally/ and follow the ones that match your environment.
Please note that you may need to restart your runtime after installation.


## 3. Prepare ManaTTS

The default experiment uses 1,000 utterances from the first ManaTTS shard. Increase `MAX_SAMPLES` for a serious training run. Existing prepared data is preserved unless `--overwrite` is included.

In [ ]:
MAX_SAMPLES = 1000
prepare_command = [
    sys.executable, 'scripts/prepare_dataset.py',
    '--max-samples', str(MAX_SAMPLES),
    '--batch-size', '16',
]
subprocess.run(prepare_command, check=True)

In [ ]:
import json

audit_path = ROOT / 'data/manatts-g2p/metadata.jsonl'
with audit_path.open(encoding='utf-8') as stream:
    for _ in range(3):
        row = json.loads(next(stream))
        print(row['text'])
        print(' ->', row['phonemes'], '\n')

## 4. Train VITS

A short run only checks that the pipeline works. High-quality synthesis requires substantially more data and training time. Checkpoints and TensorBoard logs are saved in `outputs/`.

In [ ]:
EPOCHS = 1000
BATCH_SIZE = 16
train_command = [
    sys.executable, 'scripts/train_vits.py',
    '--epochs', str(EPOCHS),
    '--batch-size', str(BATCH_SIZE),
]
subprocess.run(train_command, check=True)

## 5. Synthesize with G2P + VITS

This cell locates the newest checkpoint, converts Persian text to phonemes, and writes a WAV file.

In [ ]:
output_wav = ROOT / 'outputs/notebook_sample.wav'
subprocess.run([
    sys.executable, 'scripts/infer.py',
    '--text', 'امروز هوا بسیار خوب است.',
    '--output', str(output_wav),
], check=True)
from IPython.display import Audio
Audio(filename=str(output_wav))